In [4]:
# !pip install transformers torch accelerate
!pip install dotenv


   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 2/2 [dotenv]



### Text generation in the Pipeline function

In [ ]:
from transformers import pipeline
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv("hugging_face_token")  # Access the Hugging Face token from environment variables
generator = pipeline(
    "text-generation",
    model="gpt2",  # small and fast
    token=hf_token  # Pass the Hugging Face token
)

output = generator(
    """Question: What is the weather in bangalore in india look like?
Answer:""",
    max_new_tokens=50,
    do_sample=True,          # ✅ important
    temperature=0.7,         # ✅ controls randomness
    top_k=50,                # ✅ improves diversity
    pad_token_id=50256       # ✅ removes warning
)

print(output[0]["generated_text"])

# print(output[0]["generated_text"])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2825.10it/s]
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the weather in bangalore in india look like?
Answer: In Bangalore, the weather is very good and there is no rain.
In Bangalore, the weather is very good and there is no rain. What is the weather in bhutan in india look like?
Answer: The weather is very


### without pipeline - the acutal code

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name).to(device)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

prompt = """
Question: What is the weather in bangalore in india look like?
Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True)) 

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1872.02it/s]



Question: What is the weather in bangalore in india look like?
Answer:
A lot of people are looking for a place to live. The city has been getting quite hot lately, so it's not surprising that there have been some bad days and good nights here too! I've seen many places where you can get away from your home by walking around with friends or going out on dates (I'm sure they're all pretty cool). But if we were lucky enough at any


GPT-2 has no knowledge of current events, real-time data, or your location. It was trained on internet text up to 2019 and simply predicts likely next tokens. When you ask "What is the weather in Bangalore?" it generates statistically plausible-sounding weather text — not real data.

This is exactly why RAG exists. RAG solves this by retrieving real, current information first, then passing it to the model as context. The model's job then becomes: "given this retrieved context, generate an accurate answer" — not "hallucinate from training memory."